In [1]:
from google.colab import files
uploaded = files.upload()

Saving dataset_amb.mat to dataset_amb.mat
Saving dataset_elec.mat to dataset_elec.mat


In [3]:
import scipy.io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load Data
elec = scipy.io.loadmat('dataset_elec.mat')
amb = scipy.io.loadmat('dataset_amb.mat')

df = pd.DataFrame({
    'vdc1': elec['vdc1'].flatten(), 'vdc2': elec['vdc2'].flatten(),
    'idc1': elec['idc1'].flatten(), 'idc2': elec['idc2'].flatten(),
    'irr': amb['irr'].flatten(), 'pvt': amb['pvt'].flatten(),
    'label': amb['f_nv'].flatten()
})

# 2. Create Target Severity Values
def assign_severity(row):
    lbl = row['label']
    if lbl == 0: return 0  # Healthy
    if lbl == 1:
        # Short-Circuit Severity based on Voltage deviation
        v_diff = abs(row['vdc1'] - row['vdc2'])
        return 3 if v_diff > 60 else (2 if v_diff > 40 else 1)
    if lbl == 3:
        # Open Circuit Severity based on Potential Power Loss (Irradiance)
        irr = row['irr']
        return 3 if irr >= 800 else (2 if irr >= 400 else 1)
    return -1 # Exclude other fault labels for this specific task

df['severity_target'] = df.apply(assign_severity, axis=1)

# 3. Preprocessing (Filtering and Sampling for Balance)
# We focus on Labels 0, 1, and 3 as requested.
df_ml = df[df['severity_target'] != -1].copy()
# Downsample normal rows to 20k to balance with faults (~12k total faults)
df_final = pd.concat([
    df_ml[df_ml['severity_target'] == 0].sample(n=20000, random_state=42),
    df_ml[df_ml['severity_target'] > 0]
])

X = df_final[['vdc1', 'vdc2', 'idc1', 'idc2', 'irr', 'pvt']]
y = df_final['severity_target']

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# 5. Model Training
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluation
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 0.9939

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6000
           1       0.93      0.93      0.93        15
           2       0.97      0.97      0.97      1104
           3       0.99      0.99      0.99      2488

    accuracy                           0.99      9607
   macro avg       0.97      0.97      0.97      9607
weighted avg       0.99      0.99      0.99      9607

